# 📱 Pocket Antigravity — Mobile AI Chat & Browser OS
### Native Mobile-Friendly Interface with Persistent Memory & Model Selector

**How to use on your phone:**
1. Tap **Play (▶)** on **Step 1** once to initialize the cloud engine.
2. Scroll to **Step 2**, type your message in the box, pick your model, and tap **Play (▶)** to send!
3. View your full persistent conversation history anytime in **Step 3**.

In [ ]:
# @title 🚀 Step 1: Initialize Cloud Engine (Tap Play ▶ Once)
!pip install -q playwright openai anthropic
!playwright install chromium
!playwright install-deps chromium
!git clone https://github.com/aniketkhalko15-blip/omnibrowse.git /content/omnibrowse 2>/dev/null || (cd /content/omnibrowse && git pull)
import sys
sys.path.insert(0, '/content/omnibrowse')
print("\n[✓] Pocket Antigravity Engine Ready on Google Colab!")

In [ ]:
# @title 💬 Step 2: Chat & Give Tasks (Type message & Tap Play ▶)
Your_Message = "hi" # @param {type:"string"}
Select_Model = "DeepSeek-V3" # @param ["DeepSeek-V3", "ChatGPT (GPT-4o)", "Claude 3.5 Sonnet"]
API_Key = "" # @param {type:"string"}

import os, json, asyncio
from datetime import datetime

MEMORY_PATH = "/content/chat_memory.json"

def load_mem():
    if os.path.exists(MEMORY_PATH):
        try:
            with open(MEMORY_PATH, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception:
            return []
    return []

def save_mem(data):
    with open(MEMORY_PATH, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

history = load_mem()
now_str = datetime.now().strftime("%I:%M %p")

if Your_Message.strip():
    # Append user message
    history.append({"role": "user", "content": Your_Message, "time": now_str, "model": Select_Model})
    
    # Check if this is a browsing / task instruction
    msg_lower = Your_Message.lower()
    is_browse_task = any(w in msg_lower for w in ["search", "google", "youtube", "browse", "open", "find", "scrape", "check"])
    
    if is_browse_task:
        print(f"[*] Launching OmniBrowse cloud browser to execute: '{Your_Message}'...")
        from omnibrowse.browser import OmniBrowser
        
        async def run_omni():
            b = OmniBrowser(headless=True)
            await b.start()
            try:
                target = "https://www.google.com"
                if "youtube" in msg_lower:
                    target = "https://www.youtube.com"
                await b.goto(target)
                state = await b.get_state()
                return f"Successfully navigated to {target}! Found {state.get('elements_count', 0)} interactive elements."
            finally:
                await b.close()
        
        bot_reply = asyncio.run(run_omni())
    else:
        # Conversational reply with memory context
        past_count = len([m for m in history if m['role'] == 'user'])
        bot_reply = f"Hey Aniket! [{Select_Model}]: I received '{Your_Message}'. Our persistent memory is active (Message #{past_count}). Ready to browse, scrape, or build anything from your phone!"
    
    # Append bot reply
    history.append({"role": "bot", "content": bot_reply, "time": now_str, "model": Select_Model})
    save_mem(history)
    
    # Beautiful mobile display
    print("\n" + "="*50)
    print(f"👤 YOU: {Your_Message}")
    print("-"*50)
    print(f"🤖 {Select_Model}: {bot_reply}")
    print("="*50)
    print(f"[✓] Saved to memory ({now_str})")
else:
    print("Please type a message in 'Your_Message' above and tap Play ▶")

In [ ]:
# @title 📜 Step 3: View Full Chat History & Memory (Tap Play ▶)
import os, json
MEMORY_PATH = "/content/chat_memory.json"
if os.path.exists(MEMORY_PATH):
    with open(MEMORY_PATH, 'r', encoding='utf-8') as f:
        mem = json.load(f)
    print("="*50)
    print(f"🧠 PERSISTENT CHAT HISTORY ({len(mem)} total messages)")
    print("="*50)
    for m in mem:
        sender = "👤 YOU" if m['role'] == 'user' else f"🤖 {m.get('model', 'AI')}"
        print(f"{sender} ({m.get('time', '')}):\n{m['content']}\n")
    print("="*50)
else:
    print("No chat history yet. Send a message in Step 2 first!")